# 01 · Module 初始化与属性拦截

> **覆盖的类/函数**：`__init__`, `__setattr__`, `__getattr__`, `__delattr__`, `add_module`, `register_module`  
> **PyTorch 源码**：[torch/nn/modules/module.py](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/module.py)  
> **运行环境**：Python 3.9+, PyTorch 2.0+

---

## Part A: 源码阅读

在开始之前，先导入所需模块并定位 `module.py` 源码文件：

In [1]:
import torch
import torch.nn as nn
import inspect
from pathlib import Path

# 定位 module.py 源码路径
module_py_path = Path(inspect.getfile(nn.Module))
print(f'module.py 路径: {module_py_path}')
print(f'PyTorch 版本: {torch.__version__}')

# 查看 module.py 总行数
with open(module_py_path, 'r') as f:
    lines = f.readlines()
print(f'module.py 总行数: {len(lines)}')

module.py 路径: /Users/siwei_wang/Library/Python/3.9/lib/python/site-packages/torch/nn/modules/module.py
PyTorch 版本: 2.8.0
module.py 总行数: 3034


### A.1 `__init__` — Module 初始化了哪些数据结构？

每当你在自定义模型中写 `super().__init__()` 时，`nn.Module.__init__` 就会初始化以下内部字典：

In [2]:
# 查看 Module.__init__ 源码
init_source = inspect.getsource(nn.Module.__init__)
print(init_source)

    def __init__(self, *args, **kwargs) -> None:
        """Initialize internal Module state, shared by both nn.Module and ScriptModule."""
        torch._C._log_api_usage_once("python.nn_module")

        # Backward compatibility: no args used to be allowed when call_super_init=False
        if self.call_super_init is False and bool(kwargs):
            raise TypeError(
                f"{type(self).__name__}.__init__() got an unexpected keyword argument '{next(iter(kwargs))}'"
                ""
            )

        if self.call_super_init is False and bool(args):
            raise TypeError(
                f"{type(self).__name__}.__init__() takes 1 positional argument but {len(args) + 1} were"
                " given"
            )

        """
        Calls super().__setattr__('a', a) instead of the typical self.a = a
        to avoid Module.__setattr__ overhead. Module's __setattr__ has special
        handling for parameters, submodules, and buffers but simply calls into
     

**关键解读**：

| 属性 | 类型 | 作用 |
|------|------|------|
| `training` | `bool` | 训练/评估模式标志（`model.train()` / `model.eval()` 切换） |
| `_parameters` | `OrderedDict[str, Parameter]` | 存储可学习参数（被优化器追踪） |
| `_buffers` | `OrderedDict[str, Tensor]` | 存储不可学习状态（如 BatchNorm 的 running_mean） |
| `_non_persistent_buffers_set` | `set` | 存储非持久化 Buffer 名称（不参与 `state_dict()`） |
| `_backward_hooks` | `OrderedDict[int, Callable]` | 反向传播 hook |
| `_backward_pre_hooks` | `OrderedDict[int, Callable]` | 反向传播 pre-hook |
| `_forward_hooks` | `OrderedDict[int, Callable]` | 前向传播 hook |
| `_forward_pre_hooks` | `OrderedDict[int, Callable]` | 前向传播 pre-hook |
| `_state_dict_hooks` | `OrderedDict[int, Callable]` | `state_dict()` 相关 hook |
| `_state_dict_pre_hooks` | `OrderedDict[int, Callable]` | `state_dict()` pre-hook |
| `_load_state_dict_pre_hooks` | `OrderedDict[int, Callable]` | `load_state_dict()` pre-hook |
| `_load_state_dict_post_hooks` | `OrderedDict[int, Callable]` | `load_state_dict()` post-hook |
| `_modules` | `OrderedDict[str, Module]` | 存储子模块（形成 Module 树） |
| `_is_full_backward_hook` | `Optional[bool]` | 标记使用的 backward hook 类型 |

> 💡 **设计模式**：这是 **Composite 模式**的体现——Module 既是叶子节点（如 `nn.Linear`），也可以是组合节点（包含子 Module），统一的 `_modules` 字典递归管理整个树结构。

### A.2 `__setattr__` — 属性赋值的核心魔法

这是 Module 系统**最核心的方法**。当你写 `self.conv = nn.Conv2d(3, 64, 3)` 时，它自动检测值类型并路由到对应内部字典。

In [3]:
# 查看 __setattr__ 源码
setattr_source = inspect.getsource(nn.Module.__setattr__)
print(setattr_source)

    def __setattr__(self, name: str, value: Union[Tensor, "Module"]) -> None:
        def remove_from(*dicts_or_sets):
            for d in dicts_or_sets:
                if name in d:
                    if isinstance(d, dict):
                        del d[name]
                    else:
                        d.discard(name)

        params = self.__dict__.get("_parameters")
        if isinstance(value, Parameter):
            if params is None:
                raise AttributeError(
                    "cannot assign parameters before Module.__init__() call"
                )
            remove_from(
                self.__dict__,
                self._buffers,
                self._modules,
                self._non_persistent_buffers_set,
            )
            self.register_parameter(name, value)
        elif params is not None and name in params:
            if value is not None:
                raise TypeError(
                    f"cannot assign '{torch.typename(value)}' a

**路由规则简化版**：

```
self.x = value
    │
    ├── isinstance(value, Parameter)     → self._parameters[name] = value  ✅ 被优化器追踪
    ├── isinstance(value, Module)         → self._modules[name] = value     ✅ 参与递归遍历
    ├── name in self._buffers             → self._buffers[name] = value     ✅ 参与 to()/state_dict()
    ├── name in _parameters/_modules/...  → 先清理旧注册，再按新类型路由
    └── else                               → object.__setattr__(self, ...)  ⚠️ 普通属性，不被追踪
```

> ⚠️ **常见陷阱**：给 `self.x` 赋一个普通 `torch.Tensor` 不会自动注册为 Parameter！必须用 `nn.Parameter(tensor)` 包裹或使用 `register_parameter()`。

### A.3 `__getattr__` — 属性查找的优先级

当访问 `module.x` 时，如果普通的属性查找失败（`AttributeError`），会回退到 `__getattr__`，按优先级从内部字典中查找。

In [4]:
# 查看 __getattr__ 源码
getattr_source = inspect.getsource(nn.Module.__getattr__)
print(getattr_source)

    def __getattr__(self, name: str) -> Union[Tensor, "Module"]:
        if "_parameters" in self.__dict__:
            _parameters = self.__dict__["_parameters"]
            if name in _parameters:
                return _parameters[name]
        if "_buffers" in self.__dict__:
            _buffers = self.__dict__["_buffers"]
            if name in _buffers:
                return _buffers[name]
        if "_modules" in self.__dict__:
            modules = self.__dict__["_modules"]
            if name in modules:
                return modules[name]
        raise AttributeError(
            f"'{type(self).__name__}' object has no attribute '{name}'"
        )



**查找优先级**：`_parameters` → `_buffers` → `_modules`

这意味着如果你有一个 Parameter 和一个子 Module 同名，Parameter 会被优先返回。

### A.4 `add_module` / `register_module` — 显式添加子模块

In [5]:
# add_module 实际上是 register_module 的别名
print('add_module 源码:')
print(inspect.getsource(nn.Module.add_module))
print('\nregister_module 源码:')
print(inspect.getsource(nn.Module.register_module))

add_module 源码:
    def add_module(self, name: str, module: Optional["Module"]) -> None:
        r"""Add a child module to the current module.

        The module can be accessed as an attribute using the given name.

        Args:
            name (str): name of the child module. The child module can be
                accessed from this module using the given name
            module (Module): child module to be added to the module.
        """
        if not isinstance(module, Module) and module is not None:
            raise TypeError(f"{torch.typename(module)} is not a Module subclass")
        elif not isinstance(name, str):
            raise TypeError(
                f"module name should be a string. Got {torch.typename(name)}"
            )
        elif hasattr(self, name) and name not in self._modules:
            raise KeyError(f"attribute '{name}' already exists")
        elif "." in name:
            raise KeyError(f'module name can\'t contain ".", got: {name}')
        elif

---

## Part B: 机制分析

### B.1 为什么不能忘记 `super().__init__()`？

如果忘记调用 `super().__init__()`，`_parameters`、`_modules`、`_buffers` 等内部字典都不会被初始化。后续任何 `self.x = nn.Linear(...)` 都会因为 `__setattr__` 尝试访问 `self._modules` 而报 `AttributeError`。

In [6]:
# 演示：忘记 super().__init__() 的后果
class BadModule(nn.Module):
    def __init__(self):
        # 故意不调用 super().__init__()
        pass

try:
    bad = BadModule()
    bad.linear = nn.Linear(10, 10)  # 这里会报错！
except AttributeError as e:
    print(f'❌ 错误: {e}')
    print('\n原因: _modules 字典未被初始化，__setattr__ 尝试访问 self._modules 失败')

❌ 错误: cannot assign module before Module.__init__() call

原因: _modules 字典未被初始化，__setattr__ 尝试访问 self._modules 失败


### B.2 对比：Python 标准 `__setattr__` vs `nn.Module.__setattr__`

普通的 Python 对象：`self.x = value` 直接调用 `object.__setattr__(self, 'x', value)`  
`nn.Module` 子类：`__setattr__` 先拦截，判断类型后可能路由到内部字典

In [7]:
# 对比
class NormalClass:
    def __init__(self):
        self.x = 1  # 直接 object.__setattr__

class MyModule(nn.Module):
    def __init__(self):
        super().__init__()  # 初始化内部字典
        self.linear = nn.Linear(10, 10)  # 触发 Module.__setattr__ → 路由到 _modules
        self.normal_attr = 42              # 普通属性 → object.__setattr__

model = MyModule()
print(f'_modules 中的键: {list(model._modules.keys())}')
print(f'normal_attr: {model.normal_attr}')
print(f'linear 是普通属性吗? {"linear" in model.__dict__}')  # 应该在 _modules 而非 __dict__

_modules 中的键: ['linear']
normal_attr: 42
linear 是普通属性吗? False


---

## Part C: 动手实验

### 实验 1：追踪 `__setattr__` 的调用链

我们创建一个带日志的 Module，观察每次属性赋值到底走了哪条路径。

In [8]:
class TraceModule(nn.Module):
    """带日志的 Module，追踪 __setattr__ 调用"""
    def __init__(self):
        super().__init__()
        self.calls = []  # 正常属性（因为 calls 不是 Module/Parameter/Buffer）
    
    def __setattr__(self, name, value):
        # 记录每次调用
        value_type = type(value).__name__
        if isinstance(value, nn.Parameter):
            route = '→ _parameters'
        elif isinstance(value, nn.Module):
            route = f'→ _modules (类型: {value_type})'
        elif hasattr(self, '_buffers') and name in self._buffers:
            route = '→ _buffers (已存在)'
        else:
            route = '→ object.__setattr__'
        print(f'  __setattr__({name!r}, {type(value).__name__}) {route}')
        super().__setattr__(name, value)

print('=== 开始构建 TraceModule ===')
model = TraceModule()

=== 开始构建 TraceModule ===
  __setattr__('calls', list) → object.__setattr__


In [9]:
# 继续向 model 添加不同属性
print('--- 添加子模块 ---')
model.conv = nn.Conv2d(3, 64, 3)

print('--- 添加 Parameter ---')
model.my_param = nn.Parameter(torch.randn(10))

print('--- 添加 Buffer（通过 register_buffer）---')
model.register_buffer('running_mean', torch.zeros(64))

print('--- 赋值普通 int ---')
model.epoch = 5

print('\n=== 最终内部状态 ===')
print(f'_parameters: {list(model._parameters.keys())}')
print(f'_buffers: {list(model._buffers.keys())}')
print(f'_modules: {list(model._modules.keys())}')
print(f'__dict__ 中的普通属性: {[k for k in model.__dict__ if not k.startswith("_")]}')

--- 添加子模块 ---
  __setattr__('conv', Conv2d) → _modules (类型: Conv2d)
--- 添加 Parameter ---
  __setattr__('my_param', Parameter) → _parameters
--- 添加 Buffer（通过 register_buffer）---
--- 赋值普通 int ---
  __setattr__('epoch', int) → object.__setattr__

=== 最终内部状态 ===
_parameters: ['my_param']
_buffers: ['running_mean']
_modules: ['conv']
__dict__ 中的普通属性: ['training', 'calls', 'epoch']


### 实验 2：验证 `add_module` 的同名冲突保护 与 `__getattr__` 查找优先级

PyTorch 的 `add_module` 会阻止与已有属性同名的注册（防止 Parameter/Buffer 被子模块覆盖）。同时验证 `__getattr__` 的查找顺序。

In [10]:
# Step 1: 演示 add_module 的同名冲突保护
model = nn.Module()
model.register_parameter('weight', nn.Parameter(torch.randn(5)))

print('Step 1: 当 _parameters 中已有名为 weight 的 Parameter 时')
print(f'  _parameters keys: {list(model._parameters.keys())}')
print(f'  _modules keys: {list(model._modules.keys())}')

try:
    model.add_module('weight', nn.Linear(5, 5))
except KeyError as e:
    print(f'  ❌ add_module 拒绝: {e}')

# Step 2: 直接赋值也不能跨类型替换
print('\nStep 2: 直接赋值 nn.Linear 给已注册 Parameter 的名字')
try:
    model.weight = nn.Linear(5, 5)  # 尝试把 Module 赋给 Parameter 名
except TypeError as e:
    print(f'  ❌ __setattr__ 也拒绝: {e}')
    print(f'  原因: name 已在 _parameters 中，期望 Parameter 或 None，不接受 Module')

# 正确做法：先删除旧注册，再赋新值
print('\nStep 3: 正确做法 — 先删除旧 Parameter，再赋 Module')
del model.weight  # 这会从 _parameters 中移除
model.weight = nn.Linear(5, 5)  # 现在可以了
print(f'  _parameters keys: {list(model._parameters.keys())}')
print(f'  _modules keys: {list(model._modules.keys())}')
print(f'  type(model.weight): {type(model.weight).__name__}')

# Step 4: 演示 __getattr__ 查找顺序
print('\nStep 4: 验证 __getattr__ 查找优先级')
m = nn.Module()
m.register_parameter('a', nn.Parameter(torch.tensor([1.0])))  # 存入 _parameters
m.register_buffer('b', torch.tensor([2.0]))                      # 存入 _buffers
m.add_module('c', nn.Linear(1, 1))                               # 存入 _modules

print(f'访问 m.a (Parameter): {m.a}')  # 在 _parameters 中找到
print(f'访问 m.b (Buffer):    {m.b}')  # 在 _buffers 中找到
print(f'访问 m.c (Module):    {m.c}')  # 在 _modules 中找到

# 删除 a，验证不存在时 __getattr__ 的查找
del m.a
print(f'访问 m.a: ', end='')
try:
    print(m.a)
except AttributeError:
    print('AttributeError — 已从 _parameters 删除且 _buffers/_modules 中也没有')

print(f'\n✅ __getattr__ 查找顺序确认: _parameters → _buffers → _modules')

Step 1: 当 _parameters 中已有名为 weight 的 Parameter 时
  _parameters keys: ['weight']
  _modules keys: []
  ❌ add_module 拒绝: "attribute 'weight' already exists"

Step 2: 直接赋值 nn.Linear 给已注册 Parameter 的名字
  ❌ __setattr__ 也拒绝: cannot assign 'torch.nn.modules.linear.Linear' as parameter 'weight' (torch.nn.Parameter or None expected)
  原因: name 已在 _parameters 中，期望 Parameter 或 None，不接受 Module

Step 3: 正确做法 — 先删除旧 Parameter，再赋 Module
  _parameters keys: []
  _modules keys: ['weight']
  type(model.weight): Linear

Step 4: 验证 __getattr__ 查找优先级
访问 m.a (Parameter): Parameter containing:
tensor([1.], requires_grad=True)
访问 m.b (Buffer):    tensor([2.])
访问 m.c (Module):    Linear(in_features=1, out_features=1, bias=True)
访问 m.a: AttributeError — 已从 _parameters 删除且 _buffers/_modules 中也没有

✅ __getattr__ 查找顺序确认: _parameters → _buffers → _modules


### 实验 3：`add_module` vs 直接赋值

探究 `self.add_module(name, module)` 和 `self.name = module` 是否等价。

In [11]:
# 对比 add_module 和直接赋值
model_a = nn.Module()
model_b = nn.Module()

# 方式 1：add_module
model_a.add_module('layer1', nn.Linear(10, 10))
model_a.add_module('layer2', nn.Linear(10, 10))

# 方式 2：直接赋值（作为属性）
model_b.layer1 = nn.Linear(10, 10)
model_b.layer2 = nn.Linear(10, 10)

print('add_module 方式:')
print(f'  _modules: {list(model_a._modules.keys())}')
print(f'  named_children: {[name for name, _ in model_a.named_children()]}')
print(f'  model_a.layer1 可访问: {hasattr(model_a, "layer1")}')

print('\n直接赋值方式:')
print(f'  _modules: {list(model_b._modules.keys())}')
print(f'  named_children: {[name for name, _ in model_b.named_children()]}')
print(f'  model_b.layer1 可访问: {hasattr(model_b, "layer1")}')

# 验证 add_module 的参数是 nn.Module 时等价于直接赋值
# 事实上 add_module 内部调用了 register_module，而 register_module 调用了 __setattr__
print('\n✅ 结论: add_module(name, module) 等价于 setattr(self, name, module)')

add_module 方式:
  _modules: ['layer1', 'layer2']
  named_children: ['layer1', 'layer2']
  model_a.layer1 可访问: True

直接赋值方式:
  _modules: ['layer1', 'layer2']
  named_children: ['layer1', 'layer2']
  model_b.layer1 可访问: True

✅ 结论: add_module(name, module) 等价于 setattr(self, name, module)


### 实验 4：`__delattr__` 的行为

删除 Module 上的属性时，它是否会从内部字典中移除？

In [12]:
# 测试 __delattr__
model = nn.Module()
model.linear = nn.Linear(10, 10)
print(f'删除前 _modules: {list(model._modules.keys())}')
print(f'删除前 hasattr linear: {hasattr(model, "linear")}')

del model.linear
print(f'删除后 _modules: {list(model._modules.keys())}')
print(f'删除后 hasattr linear: {hasattr(model, "linear")}')

# 对 Parameter 同样有效
model.register_parameter('w', nn.Parameter(torch.randn(5)))
print(f'\n删除前 _parameters: {list(model._parameters.keys())}')
del model.w
print(f'删除后 _parameters: {list(model._parameters.keys())}')

删除前 _modules: ['linear']
删除前 hasattr linear: True
删除后 _modules: []
删除后 hasattr linear: False

删除前 _parameters: ['w']
删除后 _parameters: []


### 实验 5：赋值普通 Tensor 不会自动注册

这是新手最常见的陷阱。

In [13]:
# ⚠️ 常见错误：直接赋值普通 Tensor
model = nn.Module()
model.weight = torch.randn(10, 10)  # 普通 Tensor，不是 Parameter！

print(f'weight 的类型: {type(model.weight).__name__}')
print(f'_parameters 中有 weight 吗? {"weight" in dict(model.named_parameters())}')
print(f'_buffers 中有 weight 吗? {"weight" in dict(model.named_buffers())}')
print(f'weight 是普通属性吗? {"weight" in model.__dict__}')

print('\n⚠️ 这意味着:')
print('  - model.to(cuda) 不会移动 weight')
print('  - state_dict() 不会保存 weight')
print('  - optimizer 不会更新 weight')

# ✅ 正确做法
model2 = nn.Module()
model2.weight = nn.Parameter(torch.randn(10, 10))  # 包裹为 Parameter
# 或者
model2.register_parameter('bias', nn.Parameter(torch.randn(10)))
# 或者对于非可学习状态
model2.register_buffer('counter', torch.zeros(1))

print(f'\nmodel2._parameters: {list(model2._parameters.keys())}')
print(f'model2._buffers: {list(model2._buffers.keys())}')

weight 的类型: Tensor
_parameters 中有 weight 吗? False
_buffers 中有 weight 吗? False
weight 是普通属性吗? True

⚠️ 这意味着:
  - model.to(cuda) 不会移动 weight
  - state_dict() 不会保存 weight
  - optimizer 不会更新 weight

model2._parameters: ['weight', 'bias']
model2._buffers: ['counter']


---

## Part D: 小结

### 要点清单

- [x] `super().__init__()` 初始化了 4 组核心数据结构：`_parameters`、`_buffers`、`_modules`、各类 `_hooks`
- [x] `__setattr__` 是 Module 系统的核心魔法 —— 自动根据值类型路由到不同内部字典
- [x] `__getattr__` 作为后备查找，优先级：`_parameters` → `_buffers` → `_modules`
- [x] `__delattr__` 会同时清理内部字典中的注册信息
- [x] `add_module(name, m)` 等价于 `setattr(self, name, m)`，内部调用 `register_module` → `__setattr__`
- [x] 普通 `torch.Tensor` 直接赋值 `self.x = tensor` **不会**自动注册 —— 必须用 `nn.Parameter()` 或 `register_buffer()`

### 与后续 Notebook 的关联

- **Notebook 02** 将深入讲解 `register_parameter` / `register_buffer` 及 `_named_members` 的遍历引擎
- **Notebook 03** 将探索 `_modules` 构成的树结构如何通过 `named_modules()` 进行 DFS 遍历
- **Notebook 06** 将揭示 `_hooks` 字典如何参与 forward 调用链

### 延伸阅读

- [PyTorch 源码 module.py L150-L300](https://github.com/pytorch/pytorch/blob/main/torch/nn/modules/module.py#L150) — `__init__` 和 `__setattr__` 的完整实现
- [PyTorch 源码分析：Module类 (CSDN)](https://blog.csdn.net/zzxxxaa1/article/details/121037766)